In [1]:
%pip -q install ultralytics==8.4.1 lap==0.5.12 pycocotools==2.0.11 tabulate
%pip -q install --no-deps trackeval==1.3.0
!git clone https://github.com/mdmohsin212/WasteScope.git

Cloning into 'WasteScope'...
remote: Enumerating objects: 457, done.
remote: Counting objects: 100% (457/457), done.
remote: Compressing objects: 100% (256/256), done.
remote: Total 457 (delta 244), reused 374 (delta 167), pack-reused 0 (from 0)
Receiving objects: 100% (457/457), 27.35 MiB | 27.95 MiB/s, done.
Resolving deltas: 100% (244/244), done.


In [2]:
from google.colab import drive
import sys
from pathlib import Path
drive.mount("/content/drive")

REPO_DIR = Path("/content/WasteScope")
sys.path.insert(0, str(REPO_DIR))

Mounted at /content/drive


In [3]:
import hashlib
import json
import shutil
import time
from importlib.metadata import version
from types import SimpleNamespace
from zipfile import ZipFile

import cv2
import numpy as np
import pandas as pd
import torch
import ultralytics
import yaml
from itertools import product
from contextlib import ExitStack
from tqdm.auto import tqdm
from ultralytics import YOLO
from ultralytics.engine.results import Boxes
from ultralytics.trackers.byte_tracker import BYTETracker
from ultralytics.trackers.bot_sort import BOTSORT
from src.evaluation.tracking import evaluate_tracking, postprocess_tracks

Creating new Ultralytics Settings v0.0.6 file ✅ 
View Ultralytics Settings with 'yolo settings' or at '/root/.config/Ultralytics/settings.json'
Update Settings with 'yolo settings key=value', i.e. 'yolo settings runs_dir=path/to/dir'. For help see https://docs.ultralytics.com/quickstart/#ultralytics-settings.


In [4]:
BASE = Path("/content/drive/MyDrive/WasteScope")
PREPARED = BASE / "prepared/tracking"
DETECTOR = "D1_yolo26n_960"
RUN_NAME = "T2_bytetrack_postprocessed"
SPLIT = "val"

TRACKER_YAML = (
    BASE / "reports" / "tracking"
    / "T1_bytetrack_calibrated" / "val" / "tracker.yaml"
)
TRACKER_OVERRIDES = {}
POSTPROCESSOR = postprocess_tracks

POSTPROCESS_CONFIG = {
    "min_frames": 10,
    "max_gap": 5,
    "max_distance": 0.05
}

WORK = Path("/content/wastescope_tracking")
CACHE = BASE / "runs/tracking/detection_cache" / DETECTOR / f"{SPLIT}_960"
REPORT = BASE / "reports/tracking" / RUN_NAME / SPLIT

for folder in (WORK, CACHE, REPORT):
    folder.mkdir(parents=True, exist_ok=True)

data_config = json.loads((PREPARED / "data_config.json").read_text())

frames = pd.read_csv(PREPARED / data_config["frames_file"], dtype={"Sequence": str})
annotations = pd.read_csv(PREPARED / data_config["annotations_file"], dtype={"Sequence": str})

frames = frames[frames["Split"].eq(SPLIT)].sort_values(["Sequence", "Frame"])
annotations = annotations[annotations["Split"].eq(SPLIT)]

print("Sequences:", frames["Sequence"].nunique())
print("Frames:", len(frames), "| GT instances:", len(annotations))

Sequences: 13
Frames: 1480 | GT instances: 5698


In [5]:
cache_csv = CACHE / "detections.csv"
cache_json = CACHE / "metadata.json"
columns = ["Sequence", "Frame", "x1", "y1", "x2", "y2", "Score", "ClassID"]

if cache_csv.exists() and cache_json.exists():
    detections = pd.read_csv(cache_csv, dtype={"Sequence": str})
    cache_metadata = json.loads(cache_json.read_text())
else:
    local_archive = WORK / f"{SPLIT}.zip"
    shutil.copy2(PREPARED / f"{SPLIT}.zip", local_archive)
    
    weights = WORK / "best.pt"
    shutil.copy2(BASE / "runs/detect" / DETECTOR / "weights/best.pt", weights)
    model = YOLO(str(weights))
    
    rows = []
    started = time.perf_counter()
    with ZipFile(local_archive) as archive:
        for start in tqdm(range(0, len(frames), 16), desc="Caching detections"):
            batch = frames.iloc[start:start + 16]
            images = [
                cv2.imdecode(
                    np.frombuffer(archive.read(frame.ImageMember), np.uint8),
                    cv2.IMREAD_COLOR
                )
                for frame in batch.itertuples(index=False)
            ]
            
            results = model.predict(
                images,
                imgsz=960,
                batch=16,
                device=0,
                half=True,
                conf=0.001, 
                max_det=300,
                nms=False, 
                rect=True, 
                augment=False, 
                verbose=False
            )
            for frame, result in zip(batch.itertuples(index=False), results):
                rows.extend(
                    [frame.Sequence, frame.Frame, *box]
                    for box in result.boxes.data.cpu().numpy()
                )
    
    detections = pd.DataFrame(rows, columns=columns)
    cache_metadata = {
        "detector": DETECTOR,
        "checkpoint_sha256": hashlib.sha256(weights.read_bytes()).hexdigest(),
        "split": SPLIT, 
        "frames": len(frames),
        "imgsz": 960, 
        "confidence": 0.001, 
        "max_det": 300,
        "half": True, 
        "rect": True, 
        "nms": False,
        "gpu": torch.cuda.get_device_name(0),
        "ultralytics": version("ultralytics"),
        "cache_generation_seconds": time.perf_counter() - started
    }
    temp = cache_csv.with_suffix(".tmp")
    detections.to_csv(temp, index=False)
    temp.replace(cache_csv)
    cache_json.write_text(json.dumps(cache_metadata, indent=2))

print("Cached detection rows:", len(detections))

Cached detection rows: 11340


In [6]:
tracker_config = yaml.safe_load(TRACKER_YAML.read_text())
tracker_config.update(TRACKER_OVERRIDES)

detection_groups = {
    key: group[columns[2:]].to_numpy(dtype=np.float32)
    for key, group in detections.groupby(["Sequence", "Frame"])
}

display(pd.Series(tracker_config, name=RUN_NAME))

,T2_bytetrack_postprocessed
tracker_type,bytetrack
track_high_thresh,0.45
track_low_thresh,0.1
new_track_thresh,0.25
track_buffer,30
match_thresh,0.8
fuse_score,True


In [7]:
def run_tracker(config):
    rows = []
    tracker_class = {"bytetrack": BYTETracker,"botsort": BOTSORT}[config["tracker_type"]]

    with ExitStack() as stack:
        archive = None

        if config["tracker_type"] == "botsort":
            local_archive = WORK / f"{SPLIT}.zip"
            if not local_archive.exists():
                shutil.copy2(PREPARED / f"{SPLIT}.zip", local_archive)
            archive = stack.enter_context(ZipFile(local_archive))

        for sequence, sequence_frames in frames.groupby("Sequence", sort=True):
            tracker = tracker_class(SimpleNamespace(**config), frame_rate=30)

            for frame in sequence_frames.itertuples(index=False):
                data = detection_groups.get(
                    (sequence, frame.Frame),
                    np.empty((0, 6), dtype=np.float32)
                )
                boxes = Boxes(data, orig_shape=(frame.Height, frame.Width))

                image = None
                if archive is not None:
                    image = cv2.imdecode(
                        np.frombuffer(archive.read(frame.ImageMember), np.uint8),
                        cv2.IMREAD_COLOR
                    )

                for result in tracker.update(boxes, image):
                    x1, y1, x2, y2, track_id, score, class_id, _ = result
                    coordinates = np.clip(
                        [x1, y1, x2, y2],
                        [0, 0, 0, 0],
                        [frame.Width, frame.Height, frame.Width, frame.Height],
                    )
                    rows.append([
                        sequence, frame.Frame, int(track_id), int(class_id),
                        *coordinates, float(score)
                    ])

    return pd.DataFrame(rows, columns=[
        "Sequence", "Frame", "TrackID", "ClassID",
        "x1", "y1", "x2", "y2", "Score"
    ])

In [8]:
def run_experiment(run_name, config, postprocess=None, postprocess_config=None):
    report = BASE / "reports/tracking" / run_name / SPLIT
    report.mkdir(parents=True, exist_ok=True)
    postprocess_config = postprocess_config or {}

    started = time.perf_counter()
    tracks = run_tracker(config)

    if postprocess is not None:
        tracks = postprocess(tracks, frames=frames, **postprocess_config)

    elapsed = time.perf_counter() - started
    overall, per_sequence = evaluate_tracking(frames, annotations, tracks)
    overall.insert(0, "Split", SPLIT)
    overall.insert(0, "Run", run_name)

    tracks.to_csv(report / "tracks.csv", index=False)
    overall.to_csv(report / "overall.csv", index=False)
    per_sequence.to_csv(report / "per_sequence.csv")
    (report / "tracker.yaml").write_text(
        yaml.safe_dump(config, sort_keys=False)
    )

    configuration = {
        "status": "complete",
        "run": run_name,
        "split": SPLIT,
        "sequences": int(frames["Sequence"].nunique()),
        "frames": len(frames),
        "detector": cache_metadata,
        "tracker": config,
        "tracker_frame_rate_argument": 30,
        "annotations_file": data_config["annotations_file"],
        "annotations_sha256": hashlib.sha256(
            (PREPARED / data_config["annotations_file"]).read_bytes()
        ).hexdigest(),
        "manifest_sha256": data_config["manifest_sha256"],
        "evaluation": {
            "class_aware": True,
            "HOTA_IoU": "0.05:0.05:0.95",
            "IDF1_MOTA_IoU": 0.5,
           "counting": "Unique IDs per sequence; use CountEligible when present"
        },
        "postprocessing": {
            "function": postprocess.__name__ if postprocess else None,
            "parameters": postprocess_config
        },
        "tracking_wall_seconds": elapsed,
        "versions": {
            package: version(package)
            for package in (
                "ultralytics", "trackeval", "numpy", "scipy", "pandas", "lap"
            )
        },
    }

    (report / "README.md").write_text(
        f"# {run_name} — {SPLIT}\n\n"
        f"Detector: {DETECTOR}, imgsz={cache_metadata['imgsz']}.\n"
        "Settings: tracker.yaml and evaluation_config.json.\n\n"
        "HOTA, DetA, AssA, IDF1, MOTA and MAPE are percentages.\n\n"
        "Tracking metrics use all emitted boxes. Counting metrics use count-eligible IDs..\n\n"
        + overall.round(4).to_markdown(index=False)
        + "\n\n## Per-sequence results\n\n"
        + per_sequence.round(4).to_markdown()
        + "\n",
        encoding="utf-8"
    )
    (report / "evaluation_config.json").write_text(
        json.dumps(configuration, indent=2), encoding="utf-8"
    )

    display(overall.round(4))
    display(per_sequence.round(4))
    print("Report saved:", report)
    return overall, per_sequence

In [9]:
if RUN_NAME.startswith("T1_") and SPLIT == "val":
    base_config = yaml.safe_load(TRACKER_YAML.read_text())
    parameters = ["track_high_thresh", "track_low_thresh", "match_thresh"]
    settings = list(product(
        [0.25, 0.35, 0.45],
        [0.05, 0.10],
        [0.70, 0.80, 0.90]
    ))

    records = []
    for values in tqdm(settings, desc="T1 calibration"):
        changes = dict(zip(parameters, values))
        candidate_config = {**base_config, **changes}
        candidate_tracks = run_tracker(candidate_config)
        metrics, _ = evaluate_tracking(frames, annotations, candidate_tracks)
        records.append({**changes, **metrics.iloc[0].to_dict()})

    calibration = pd.DataFrame(records)
    default_rows = np.logical_and.reduce([
        calibration[name].eq(base_config[name]).to_numpy()
        for name in parameters
    ])
    baseline = calibration.loc[default_rows].iloc[0]

    calibration["Eligible"] = (
        calibration["MAE"].le(baseline["MAE"])
        & calibration["IDF1"].ge(baseline["IDF1"])
    )
    best = calibration[calibration["Eligible"]].sort_values(
        ["HOTA", "IDF1", "MAE"], ascending=[False, False, True]
    ).iloc[0]

    tracker_config = {
        **base_config,
        **{name: float(best[name]) for name in parameters},
    }

    calibration.to_csv(REPORT / "calibration.csv", index=False)
    (REPORT / "selection_rule.json").write_text(json.dumps({
        "rule": "Highest HOTA with MAE <= T0 and IDF1 >= T0",
        "tuned_on": "val",
        "selected_config": tracker_config
    }, indent=2))


overall, per_sequence = run_experiment(
    RUN_NAME, tracker_config, POSTPROCESSOR, POSTPROCESS_CONFIG
)

,Run,Split,HOTA,DetA,AssA,IDF1,MOTA,IDSW,GT,RawTracks,Predicted,MAE,RMSE,SAD,MAPE
0,T2_bytetrack_postprocessed,val,62.2186,57.3582,67.7628,78.7734,63.4082,17,56,67,52,1.0769,1.6641,14,28.9835


,HOTA,DetA,AssA,IDF1,MOTA,IDSW,GT,RawTracks,Predicted,CountError,AbsCountError
Sequence,,,,,,,,,,,
0010_1,90.7317,90.7317,90.7317,100.0000,100.0000,0,1,1,1,0,0
0021_1,0.0000,0.0000,0.0000,0.0000,0.0000,0,1,0,0,-1,1
0024_1,67.7812,65.1741,70.8343,84.9306,70.3190,0,8,10,7,-1,1
0029_1,65.7050,58.9696,73.2603,82.4337,66.7614,1,6,9,8,2,2
0039_1,72.9282,71.4229,74.5358,89.4536,80.8252,2,5,6,5,0,0
0063_1,61.1998,60.0477,62.8067,77.2532,57.8348,4,7,10,8,1,1
0069_1,80.7728,80.7728,80.7728,95.8333,92.0000,0,1,1,1,0,0
0075_1,40.6979,34.6988,47.8464,59.9402,35.6511,1,5,7,5,0,0
0094_1,42.0834,48.1211,36.8662,70.0000,61.1111,2,2,3,1,-1,1


Report saved: /content/drive/MyDrive/WasteScope/reports/tracking/T2_bytetrack_postprocessed/val
